<a href="https://colab.research.google.com/github/Arsh-Vora/Arxiv-PageRank-Analysis/blob/dev/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import itertools

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)


100%|██████████| 1.58G/1.58G [00:16<00:00, 106MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/276


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OptimizedPageRank") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()


In [4]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("authors_parsed", ArrayType(ArrayType(StringType())), True)
])

raw_df = spark.read.json(path + "/arxiv-metadata-oai-snapshot.json", schema=schema)

df_flattened = raw_df.withColumn("author_full", F.explode("authors_parsed")) \
    .withColumn("author_name", F.concat_ws(", ", F.col("author_full"))) \
    .select(F.col("id").alias("paper_id"), "author_name")


authors_mapping = df_flattened.select("author_name").distinct() \
    .withColumn("author_id", F.monotonically_increasing_id())

bipartite_df = df_flattened.join(authors_mapping, "author_name") \
    .select("author_id", "paper_id")

bipartite_df.show(5)


+---------+----------+
|author_id|  paper_id|
+---------+----------+
|    63441| 1409.0865|
|    55699| 1306.5960|
|    55699| 1306.6489|
|    55699| 1307.6542|
|   209487|2311.12868|
+---------+----------+
only showing top 5 rows


In [8]:
paper_weights = bipartite_df.groupBy("paper_id").count() \
    .withColumn("weight", 1.0 / F.col("count")) \
    .select("paper_id", "weight")

author_out_degree = bipartite_df.groupBy("author_id").count() \
    .withColumn("out_degree", F.col("count"))

network_structure = bipartite_df.join(paper_weights, "paper_id").select("author_id", "paper_id", "weight")
network_structure.cache()


network_structure.cache()

num_authors = authors_mapping.count()
print(f"Total Authos:{num_authors}")

Total Authos:2368706


In [ ]:
ranks = authors_mapping.select("author_id").withColumn("rank", F.lit(1.0))

iterations = 10
damping_factor = 0.85

for i in range(iterations):
  paper_contributions = network_structure.join(ranks,"author_id")\
  .withColumn("contribution",F.col("rank")*F.col("weight"))

  new_ranks_raw = paper_contributions.groupBy("author_id")\
  .agg(F.sum("contribution").alias("received_rank"))

  ranks = new_ranks_raw.withColumn("rank",F.lit((1-damping_factor))+(F.lit(damping_factor)*F.col("received_rank")))

  ranks = ranks.checkpoint() if spark.sparkContext.getCheckpointDir() else ranks.cache()

  print(f"iteration{i+1} complete")
  ranks.show(5)

iteration1 complete
